# TRUEFRAME small closed-set demo training (Google Colab T4)

This demo trains on the exact images stored in `gimg/` and `aiimg/` and reuses those images during checkpoint selection. It is intentionally a memorization demo; do not present its metrics as accuracy on unseen photos. Select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
from pathlib import Path
import subprocess
repo_dir = Path('/content/trueframe')
if (repo_dir / '.git').exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/chsrikar/trueframe.git', str(repo_dir)], check=True)
%cd /content/trueframe
!pip install -q -r requirements-colab.txt

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
import os, torch
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(torch.cuda.get_device_name(0))

# Upload the existing gimg/ and aiimg/ folders to My Drive/trueframe-demo/.
os.environ['TRUEFRAME_DATA_DIR'] = '/content/drive/MyDrive/trueframe-demo'
os.environ['TRUEFRAME_DEMO_MODE'] = '1'
run_dir = Path('/content/drive/MyDrive/trueframe-runs') / datetime.now().strftime('%Y%m%d-%H%M%S')
run_dir.mkdir(parents=True, exist_ok=True)
os.environ['TRUEFRAME_MANIFEST_DIR'] = str(run_dir / 'manifests')
os.environ['TRUEFRAME_CHECKPOINT_DIR'] = str(run_dir / 'checkpoints')
os.environ['TRUEFRAME_OUTPUT_DIR'] = str(run_dir / 'outputs')
for key in ('TRUEFRAME_MANIFEST_DIR', 'TRUEFRAME_CHECKPOINT_DIR', 'TRUEFRAME_OUTPUT_DIR'):
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)

## Add the demo images to Google Drive

Upload your existing `gimg/` folder (real/mobile photos) and `aiimg/` folder (ChatGPT-generated images) under `My Drive/trueframe-demo/`, so the paths are `My Drive/trueframe-demo/gimg/` and `My Drive/trueframe-demo/aiimg/`. The notebook labels images by folder name. Personal image folders are kept out of Git. The checkpoint is for demonstrating on these known images only; because the same images train and select the model, this run cannot establish accuracy on new photos.

In [ ]:
# Check folders and enable HEIC decoding before training.
%pip -q install pillow-heif
import pillow_heif
pillow_heif.register_heif_opener()
from pathlib import Path
data_dir = Path(os.environ['TRUEFRAME_DATA_DIR'])
# Patch the existing dataset module in this Colab clone so worker processes can read HEIC too.
dataset_file = Path('dataset.py')
dataset_source = dataset_file.read_text()
registration = "\nimport pillow_heif\npillow_heif.register_heif_opener()\n"
if 'register_heif_opener' not in dataset_source:
    dataset_source = dataset_source.replace('from PIL import Image', 'from PIL import Image' + registration, 1)
    dataset_file.write_text(dataset_source)
for folder in ('gimg', 'aiimg'):
    path = data_dir / folder
    if not path.is_dir():
        raise FileNotFoundError(f'Upload the {folder}/ folder to {path}')
    images = [x for x in path.rglob('*') if x.is_file() and x.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff', '.heic', '.heif'}]
    if not images:
        raise RuntimeError(f'No supported images found in {path}')
    print(folder, len(images), 'images')


In [ ]:
# Create closed-set demo manifests directly from the uploaded folder labels.
import json
import pandas as pd
from PIL import Image
records = []
supported = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff', '.heic', '.heif'}
for folder, label, generator in (('gimg', 'genuine', 'camera_photo'), ('aiimg', 'ai_generated', 'chatgpt')):
    for image_path in sorted((data_dir / folder).rglob('*')):
        if not image_path.is_file() or image_path.suffix.lower() not in supported:
            continue
        try:
            with Image.open(image_path) as image:
                image.verify()
        except Exception as exc:
            raise RuntimeError(f'Could not decode {image_path}: {exc}') from exc
        records.append({'filepath': str(image_path.resolve()), 'label': label, 'source': folder, 'generator': generator})
df = pd.DataFrame(records)
if set(df['label']) != {'genuine', 'ai_generated'}:
    raise RuntimeError('Both gimg/ and aiimg/ must contain readable images.')
manifest_dir = Path(os.environ['TRUEFRAME_MANIFEST_DIR'])
manifest_dir.mkdir(parents=True, exist_ok=True)
for name in ('manifest_full.csv', 'manifest_train.csv', 'manifest_val.csv', 'manifest_test.csv'):
    df.to_csv(manifest_dir / name, index=False)
protocol = {'name': 'closed_set_demo_memorization', 'train_validation_test': 'same exact images', 'valid_generalization_metrics': False, 'warning': 'Demo replay only; do not report as unseen-image accuracy.'}
(manifest_dir / 'split_protocol.json').write_text(json.dumps(protocol, indent=2))
print(df.groupby(['label', 'source']).size().to_string())
print(json.dumps(protocol, indent=2))

In [ ]:
# Train a demo checkpoint. MixUp is off so the model can memorize the exact examples.
manifest_dir = Path(os.environ['TRUEFRAME_MANIFEST_DIR'])
subprocess.run([
    sys.executable, 'train.py',
    '--train-manifest', str(manifest_dir / 'manifest_train.csv'),
    '--val-manifest', str(manifest_dir / 'manifest_val.csv'),
    '--batch-size', '4', '--num-workers', '0', '--seed', '42',
    '--phase1-epochs', '5', '--phase2-epochs', '40', '--patience', '20', '--mixup-alpha', '0'
], check=True)

In [ ]:
# Save a convenient copy in Drive and download the checkpoint for the local interface.
import shutil
from google.colab import files
best_checkpoint = Path(os.environ['TRUEFRAME_CHECKPOINT_DIR']) / 'best_model.pth'
if not best_checkpoint.exists():
    raise FileNotFoundError(f'Training did not produce {best_checkpoint}')
drive_checkpoint = Path(os.environ['TRUEFRAME_DATA_DIR']) / 'best_model.pth'
shutil.copy2(best_checkpoint, drive_checkpoint)
print('Demo-only checkpoint saved to Drive:', drive_checkpoint)
files.download(str(best_checkpoint))